# Notebook 07 — The Euler Method

**Module:** 02 — Mathematics for Biology  
**Notebook:** 07 of 12  
**Prerequisites:** NB06  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Most ODEs in computational biology cannot be solved analytically. The Lotka-Volterra
system (NB09), the SIR model, most mechanistic gene regulatory network models —
none have closed-form solutions. The Euler method is the simplest numerical ODE
solver: it approximates the true trajectory by stepping forward in time using the
derivative as a local linear approximation. Understanding it from scratch makes
every more sophisticated solver (RK4, LSODA, adaptive steppers) interpretable.

---
## Step 2 — Intuition

The Euler method says: at each time step, take the current position and move in the
direction the ODE points. Over a small step $h$:
$$N(t + h) \approx N(t) + h \cdot f(N(t), t)$$

It is the ODE equivalent of 'walk in the direction you're currently facing, then
check the direction again.' The smaller $h$, the more often you check, the more
accurate the path.

---
## Step 3 — Biological Background

Euler method is used in:
- **Stochastic simulations:** the Gillespie algorithm for discrete reaction systems
  is a stochastic analogue
- **Agent-based models:** each agent's state updates by discrete steps
- **Sensitivity analysis:** finite differences (NB02) are Euler steps applied to
  parameter perturbations

In practice, `scipy.integrate.solve_ivp` uses adaptive-step RK45 by default and
is almost always preferable to a hand-rolled Euler. But you cannot debug or trust a
black-box solver without understanding what it is approximating.

---
## Step 4 — Mathematical Explanation

**Taylor expansion justification:**

Expand $N(t+h)$ around $t$:
$$N(t+h) = N(t) + h N'(t) + \frac{h^2}{2} N''(t) + \ldots$$

The Euler method keeps only the first two terms. The dropped term $\frac{h^2}{2}N''(t)$
is the **local truncation error** — the error at each single step.

**Global error:** with $n = T/h$ steps, the total error is $O(h)$ — halve $h$, halve
the error.

**Stability:** for the test equation $dN/dt = \lambda N$ with $\lambda < 0$, the
Euler method is stable only when $|1 + h\lambda| \leq 1$, i.e., $h \leq 2/|\lambda|$.
Large step sizes on stiff equations cause oscillating, diverging solutions.

---
## Step 5 — Computational Explanation

**Euler algorithm:**
```
t = t0, N = N0
while t < T:
    N = N + h * f(N, t)
    t = t + h
```

**Vectorised implementation for systems:** if $y$ is a vector (multiple species),
`f(y, t)` returns a vector and the update is element-wise: $y \leftarrow y + h \cdot f(y, t)$.

**Error analysis:** compare Euler trajectory to analytical solution (or high-accuracy
`solve_ivp`) as a function of $h$. Plot error vs. $h$ on a log-log scale — slope
should be approximately 1 (first-order method).

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Euler method from scratch
def euler(f, y0: np.ndarray, t_span: tuple, h: float):
    """
    Forward Euler method for dy/dt = f(y, t).

    Parameters
    ----------
    f : callable
        RHS function f(y, t); returns array of same shape as y.
    y0 : np.ndarray
        Initial state vector.
    t_span : (t0, T)
        Integration interval.
    h : float
        Time step.

    Returns
    -------
    t_arr : np.ndarray  shape (n+1,)
    y_arr : np.ndarray  shape (n+1, len(y0))
    """
    t0, T = t_span
    n = int(np.ceil((T - t0) / h))
    t_arr = np.linspace(t0, t0 + n * h, n + 1)
    y_arr = np.empty((n + 1, len(y0)))
    y_arr[0] = y0
    for i in range(n):
        y_arr[i+1] = y_arr[i] + h * np.asarray(f(y_arr[i], t_arr[i]))
    return t_arr, y_arr

# Test: logistic growth
r, K, N0 = 0.5, 100.0, 5.0
f_logistic = lambda y, t: np.array([r * y[0] * (1 - y[0] / K)])

t_euler, y_euler = euler(f_logistic, [N0], (0, 20), h=0.5)
print(f"Euler (h=0.5) final N: {y_euler[-1, 0]:.4f}")
print(f"Analytical final N:    {K / (1 + (K/N0 - 1) * np.exp(-r * 20)):.4f}")

In [ ]:
# Cell 6.2 — Error vs. step size analysis
def logistic_analytical(t, r, K, N0):
    return K / (1 + (K/N0 - 1) * np.exp(-r * t))

T_end = 20.0
h_vals = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.01]
errors = []

N_true = logistic_analytical(T_end, r, K, N0)
for h in h_vals:
    _, y = euler(f_logistic, [N0], (0, T_end), h)
    errors.append(abs(y[-1, 0] - N_true))

print(f"{'h':>8} {'|Error|':>12} {'Ratio (consecutive)':>22}")
print("-" * 45)
for i, (h, err) in enumerate(zip(h_vals, errors)):
    ratio = errors[i-1] / err if i > 0 else float('nan')
    print(f"  {h:>6.3f}   {err:>10.4e}   {ratio:>10.2f}")

In [ ]:
# Cell 6.3 — Instability demonstration on a stiff-like decay equation
# dN/dt = -10*N  (fast decay: lambda = -10)
f_fast = lambda y, t: np.array([-10.0 * y[0]])
N0_stiff = [1.0]
t_exact = np.linspace(0, 1, 200)
N_exact = np.exp(-10 * t_exact)

results = {}
for h, label in [(0.15, "Stable (h=0.15)"), (0.25, "Unstable (h=0.25)")]:
    t_e, y_e = euler(f_fast, N0_stiff, (0, 1), h)
    results[label] = (t_e, y_e[:, 0])
    print(f"{label}: final |N| = {abs(y_e[-1, 0]):.2e}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Three panels: trajectories, convergence, stability
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Euler at different h vs. analytical
ax = axes[0]
t_an = np.linspace(0, 20, 300)
ax.plot(t_an, logistic_analytical(t_an, r, K, N0), color="black", lw=2, label="Analytical")
for h, color in [(2.0, "tomato"), (0.5, "orange"), (0.1, "steelblue")]:
    t_e, y_e = euler(f_logistic, [N0], (0, 20), h)
    ax.plot(t_e, y_e[:, 0], "--", color=color, lw=1.5, label=f"Euler h={h}")
ax.set_xlabel("t"); ax.set_ylabel("N(t)")
ax.set_title("Euler accuracy vs. step size")
ax.legend(fontsize=8)

# Panel 2: error vs. h (log-log)
ax = axes[1]
ax.loglog(h_vals, errors, "o-", color="steelblue", lw=2)
# Reference slope-1 line
h_ref = np.array([h_vals[0], h_vals[-1]])
ax.loglog(h_ref, errors[0] * h_ref / h_vals[0], "k--", lw=1, label="Slope 1 (reference)")
ax.set_xlabel("h (step size)"); ax.set_ylabel("|Error at t=T|")
ax.set_title("First-order convergence")
ax.legend()

# Panel 3: instability
ax = axes[2]
ax.plot(t_exact, N_exact, color="black", lw=2, label="Exact")
for label, (t_e, y_e) in results.items():
    color = "steelblue" if "Stable" in label else "tomato"
    ax.plot(t_e, y_e, "--", color=color, lw=1.5, label=label)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel("t"); ax.set_ylabel("N")
ax.set_title("Euler instability for large h")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `euler` and verify it on the mRNA ODE from NB02:
   $dM/dt = \alpha - \delta M$ with $\alpha = 5$, $\delta = 0.5$, $M(0) = 0$.
   Compare against the analytical solution $M(t) = (\alpha/\delta)(1 - e^{-\delta t})$.
2. For the logistic ODE with $r=0.5, K=100, N_0=5$, find the largest $h$ for which
   the Euler method remains within 5% of the analytical answer at $t=20$.
3. Explain in one paragraph why a first-order method has global error $O(h)$ even though
   the local truncation error is $O(h^2)$.
4. Apply `euler` to the SIR model:
   $dS/dt = -\beta SI/N$, $dI/dt = \beta SI/N - \gamma I$, $dR/dt = \gamma I$
   with $\beta = 0.3$, $\gamma = 0.1$, $N = 1000$, $I(0) = 1$, $S(0) = 999$.
   Compare $h=1$ vs $h=0.1$.

---
## Quiz — Active Recall

1. Write the Euler update equation from memory.
2. What is the local truncation error of the Euler method? How does it differ from global error?
3. What does 'first-order method' mean in the context of ODE solvers?
4. What condition makes the Euler method unstable for $dN/dt = \lambda N$ with $\lambda < 0$?
5. If halving $h$ approximately halves the error, what order is the method? Justify.

---
## Reflection

**Date completed:** ____________________

> *[Could you implement euler() from memory in an interview? What is the instability condition you'd quote?]*

---
*Next: `08_runge_kutta_rk4.ipynb`*